# AXI test 7: the confirmation test (Gemini 3.5 Flash Lite)
Every earlier test was read on a few episodes and a few seeds. This one repeats the long-run findings on **fresh tasks nobody has looked at**, with **30 episodes instead of 4 to 6**, and prints an error margin next to every number. The thresholds were written down first (`docs/CONFIRM_TEST.md`). It cannot prove the framework in general; it is where the first findings should break if they were an accident.

**What to know**
* **No GPU.** Use the free CPU runtime (Runtime menu -> *Change runtime type* -> CPU). Nothing is downloaded.
* **How long.** About 3,200 calls in all (560 single requests and 2 x 30 long episodes). The free key allows about 480 calls a day, so on the free key it takes about **a week**: every cell stops by itself, everything finished is saved on your Google Drive, and you simply run **Cells 2 to 5 again** the next day and then the cell that stopped. If billing is switched on for your key, set `PAID = True` in Cell 5 and it runs in one sitting (a few hours at most). The cost for a Flash-class model should be small; the notebook prints the tokens used so you can see.
* **Your key** lives in Colab Secrets (Cell 1). It is never printed or saved in any file. Never paste it in the chat.

Run each cell **one at a time, top to bottom**, wait for the green tick before the next one.

### Cell 1: put your key into Colab Secrets (once; no code to run)
1. In Google AI Studio (`aistudio.google.com`), open **API Keys** in the left menu and copy your key.
2. Here in Colab, tap the **key icon** in the left bar (*Secrets*) -> **Add new secret**.
3. Name: `GEMINI_API_KEY` (exactly). Value: paste the key. Switch **Notebook access** on (it is per notebook, so do it again for this one).
Do not paste the key into a cell or into the chat.

### Cell 2: upload the code. A button 'Choose files' appears: pick `axi-colab-confirm.zip`

In [ ]:
from google.colab import files
up = files.upload()
name = list(up)[0]
!rm -rf /content/axi && mkdir -p /content/axi && unzip -q -o "{name}" -d /content/axi
!ls /content/axi/python

### Cell 3: install the Gemini library (about 20 seconds)

In [ ]:
!pip -q install -U google-genai

### Cell 4: connect Google Drive (a pop-up asks permission: tap Connect / Allow). This is where your progress and every reply are kept

In [ ]:
import sys, os
sys.path.insert(0, "/content/axi/python")
from colab.colab_persist import open_store
store = open_store("/content/out", "axi_confirm")

### Cell 5: connect to the model and check it with 6 requests (about 1 minute). It prints SMOKE OK or STOP. If it says STOP, do not run the next cells: send me the printout
`PAID = False` is for the free key. Change it to `True` only if billing is on for your key. To test **another model** instead, change `WANTED` to its name (for example `gemini-3.8-flash`) and set `PAID = True`; it then keeps its own files and nothing of the first model is mixed in.

In [ ]:
import json, time
from google.colab import userdata
from axi.experiments.api_llm import QuotaStop, ServiceBusy, build_gemini_llm
from axi.experiments.confirm_test import (EPISODES, N_SINGLE, SEED_SINGLE, SEED_LONG, run_single, run_long_confirm, single_set, print_confirm_long, print_verdict, verdict_confirm)
from axi.experiments.scope_test import print_scope
from axi.experiments.bigger_model import mean_seconds
from axi.experiments.api_test import smoke_report

try: KEY = userdata.get("GEMINI_API_KEY")
except Exception: raise SystemExit("No key found. Do Cell 1: add a Secret named GEMINI_API_KEY and switch Notebook access on.")
WANTED = "gemini-3.5-flash-lite"
PAID = False          # True only if billing is switched on for this key
DAILY_FREE = 480
SLUG = WANTED.replace(".", "_")
CACHE = f"axi_confirm_cache_{SLUG}.jsonl"
print("reply file:", store.restore(CACHE))
llm, MODEL = build_gemini_llm(KEY, WANTED, store.path(CACHE), daily_limit=(100000 if PAID else DAILY_FREE), min_interval=(1.0 if PAID else 4.5), on_write=lambda: store.persist(CACHE))
RESET = "The free quota resets at midnight Pacific time: 12:30 pm in India until 1 November, 1:30 pm after. Then open this notebook again, run Cells 2 to 5, and run the cell that stopped again."
BUSY = "This is Google's server being busy. Wait at least 30 minutes, then run THIS SAME CELL once (if this page is still open, you do not need the earlier cells). Do NOT run it again and again: Google counts failed tries against the free daily limit."
T0 = time.time()
def prog(i, n): print(f"  {i}/{n} done   {time.time() - T0:6.0f} s   calls today: {llm.calls_today()}" + ("" if store.persistent else "   (NOT saved to Drive)"), end="\r")
def save(res, path):
    json.dump(res, open(path, "w"), indent=1); return path
def go(fn):
    try: return fn()
    except QuotaStop as e:
        print("\nSTOPPED FOR NOW: " + str(e)); print(BUSY if isinstance(e, ServiceBusy) else RESET); return None
res_s_norule = res_s_rule = res_l_norule = res_l_rule = None
smoke = go(lambda: run_single(llm, True, "/content/smoke_confirm.jsonl", meta=dict(model=MODEL, smoke=True), tasks=single_set()[:6]))
if smoke:
    for r in smoke["rows"]: print(f"{r['kind']:14s} {r['seconds']:5.1f} s  {r['text'][:100]!r}")
    print("totals so far:", llm.usage())
    ok, msg = smoke_report(smoke["rows"]); print("\n" + msg)

### Cell 6: PART 1, no rule in the prompt. 280 single requests, one at a time (about 25 minutes on the free key; about 5 minutes if billing is on)

In [ ]:
T0 = time.time()
name = f"axi_confirm_single_norule_{SLUG}"
print("progress file:", store.restore(name + ".jsonl"))
res_s_norule = go(lambda: run_single(llm, False, store.path(name + ".jsonl"), meta=dict(model=MODEL, seed=SEED_SINGLE), progress=store.progress(name + ".jsonl", prog)))
if res_s_norule:
    print(); print_scope(res_s_norule["summary"], "PART 1, no rule, " + MODEL)
    print("mean seconds per reply:", round(mean_seconds(res_s_norule["rows"]), 2), "  calls today:", llm.calls_today())
    files.download(save(res_s_norule, store.path(name + ".json"))); store.persist(name + ".json")

### Cell 7: PART 1, rule stated in the prompt. 280 single requests (about 25 minutes on the free key)

In [ ]:
assert res_s_norule, "The cell above stopped for today. Run it again first."
T0 = time.time()
name = f"axi_confirm_single_rule_{SLUG}"
print("progress file:", store.restore(name + ".jsonl"))
res_s_rule = go(lambda: run_single(llm, True, store.path(name + ".jsonl"), meta=dict(model=MODEL, seed=SEED_SINGLE), progress=store.progress(name + ".jsonl", prog)))
if res_s_rule:
    print(); print_scope(res_s_rule["summary"], "PART 1, rule stated, " + MODEL)
    print("mean seconds per reply:", round(mean_seconds(res_s_rule["rows"]), 2), "  calls today:", llm.calls_today())
    files.download(save(res_s_rule, store.path(name + ".json"))); store.persist(name + ".json")

### Cell 8: PART 2, no rule. 30 episodes x 30 requests, 3 policies (about 1,300 calls: 2 to 3 days on the free key, about an hour if billing is on; it stops for the day and carries on)

In [ ]:
assert res_s_rule, "The cell above stopped for today. Run it again first."
T0 = time.time()
name = f"axi_confirm_long_norule_{SLUG}"
print("progress file:", store.restore(name + ".jsonl"))
res_l_norule = go(lambda: run_long_confirm(llm, False, store.path(name + ".jsonl"), meta=dict(model=MODEL), progress=store.progress(name + ".jsonl", prog)))
if res_l_norule:
    print(); print_confirm_long(res_l_norule["analysis"], "PART 2, no rule, " + MODEL)
    print("calls today:", llm.calls_today(), "  totals:", llm.usage())
    files.download(save(res_l_norule, store.path(name + ".json"))); store.persist(name + ".json")

### Cell 9: PART 2, rule stated. 30 episodes x 30 requests (about 1,300 calls). Prints every reading-rule line at the end

In [ ]:
assert res_l_norule, "The cell above stopped for today. Run it again first."
T0 = time.time()
name = f"axi_confirm_long_rule_{SLUG}"
print("progress file:", store.restore(name + ".jsonl"))
res_l_rule = go(lambda: run_long_confirm(llm, True, store.path(name + ".jsonl"), meta=dict(model=MODEL), progress=store.progress(name + ".jsonl", prog)))
if res_l_rule:
    print(); print_confirm_long(res_l_rule["analysis"], "PART 2, rule stated, " + MODEL)
    single = {False: res_s_norule["summary"], True: res_s_rule["summary"]}
    long_ = {False: res_l_norule["analysis"], True: res_l_rule["analysis"]}
    print_verdict(verdict_confirm(single, long_, {False: res_s_norule["rows"], True: res_s_rule["rows"]}, {False: res_l_norule["rows"], True: res_l_rule["rows"]}), "all four parts")
    print("calls today:", llm.calls_today(), "  totals:", llm.usage())
    files.download(save(res_l_rule, store.path(name + ".json"))); store.persist(name + ".json")
    files.download(store.path(CACHE))

### Done
Files downloaded (the model name is in each file name): `axi_confirm_single_norule_*.json`, `axi_confirm_single_rule_*.json`, `axi_confirm_long_norule_*.json`, `axi_confirm_long_rule_*.json` and `axi_confirm_cache_*.jsonl`. Upload them to the chat, with a screenshot of the printed tables and the 'reading rule' lines. Do not paste any keys anywhere.